In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
print('✅ T4 GPU ready!' if 'T4' in result.stdout else '⚠️ Check runtime type (should be T4 GPU)')

Mon Apr  6 05:24:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
print("Installing dependencies... (usually 2-8 minutes)")

# Stable installation for Colab T4 in 2026
!pip install -q huggingface_hub transformers datasets numpy pandas

# Best working llama-cpp-python for T4
print("Installing llama-cpp-python with CUDA...")
!pip install llama-cpp-python[server] --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 -q

print("\n✅ Dependencies installed!")

Installing dependencies... (usually 2-8 minutes)
Installing llama-cpp-python with CUDA...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 GB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.0 MB/s eta 0:00:00

✅ Dependencies installed!


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os

# EDIT THIS PATH if needed
QUANTIZED_PATH = "/content/drive/MyDrive/models/qwen2.5-7b-instruct-Q4_K_M.gguf"

if os.path.exists(QUANTIZED_PATH):
    size = os.path.getsize(QUANTIZED_PATH) / (1024**3)
    print(f"✅ Quantized model found: {QUANTIZED_PATH} ({size:.2f} GB)")
else:
    print(f"❌ Model not found at {QUANTIZED_PATH}")
    print("Please upload your Q4_K_M GGUF to Google Drive and update the path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Quantized model found: /content/drive/MyDrive/models/qwen2.5-7b-instruct-Q4_K_M.gguf (4.36 GB)


In [5]:
from huggingface_hub import snapshot_download

ORIGINAL_DIR = "/content/qwen-original"
os.makedirs(ORIGINAL_DIR, exist_ok=True)

print("Downloading original Qwen2.5-7B-Instruct (BF16, ~14 GB)...")
snapshot_download(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    local_dir=ORIGINAL_DIR,
    ignore_patterns=["*.pt", "*.bin"]  # keep safetensors
)
print(f"✅ Original model downloaded to {ORIGINAL_DIR}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

✅ Original model downloaded to /content/qwen-original


In [7]:
# Cell 5: Load models + Speed test (No perplexity to avoid errors)

from llama_cpp import Llama
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import time

print("Loading Q4_K_M GGUF model...")
llm_q = Llama(
    model_path=QUANTIZED_PATH,
    n_gpu_layers=-1,      # Use T4 GPU fully
    n_ctx=4096,
    verbose=False
)

print("✅ Q4_K_M model loaded")

# === Quick Speed Test ===
print("\nRunning speed test on Q4_K_M...")
start = time.time()
resp = llm_q(
    "Write a short professional email declining a meeting request politely.",
    max_tokens=150,
    temperature=0.7
)
elapsed = time.time() - start
generated_text = resp["choices"][0]["text"]
tokens_generated = len(generated_text.split())
speed = tokens_generated / elapsed if elapsed > 0 else 0

print(f"Q4_K_M Speed: **{speed:.1f} tokens/sec**")
print(f"Generated {tokens_generated} tokens in {elapsed:.1f} seconds\n")
print("Sample output:\n", generated_text[:500] + "..." if len(generated_text) > 500 else generated_text)

# === Load Original BF16 Model ===
print("\nLoading original BF16 model (this may take 1-2 minutes)...")
tokenizer = AutoTokenizer.from_pretrained(ORIGINAL_DIR, trust_remote_code=True)
model_hf = AutoModelForCausalLM.from_pretrained(
    ORIGINAL_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
pipe = pipeline(
    "text-generation",
    model=model_hf,
    tokenizer=tokenizer,
    max_new_tokens=150
)

print("✅ Original BF16 model loaded")

Loading Q4_K_M GGUF model...


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Q4_K_M model loaded

Running speed test on Q4_K_M...
Q4_K_M Speed: **24.8 tokens/sec**
Generated 125 tokens in 5.0 seconds

Sample output:
  Subject: Declining Meeting Request

Dear [Recipient's Name],

I hope this message finds you well.

Thank you for your recent meeting request on [Date] at [Time]. I appreciate the opportunity to discuss [briefly mention the topic if relevant], and I value our collaboration.

Unfortunately, my schedule is already quite full this week, and I won't be able to make it to the meeting on the proposed date. I would, however, be more than willing to schedule a meeting at a different time that works bett...

Loading original BF16 model (this may take 1-2 minutes)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Original BF16 model loaded


In [8]:
PROMPTS = [
    "Draft a polite professional email to a client explaining that the project delivery will be delayed by 3 days due to unexpected supply chain issues.",
    "A team member missed an important deadline. Write a constructive feedback message that is professional but firm.",
    "Summarize this meeting note in 3 bullet points: We discussed Q2 sales targets, marketing budget increase of 15%, and new CRM tool evaluation.",
    "Is increasing debt to fund expansion a good idea if revenue is growing 8% yearly? Give short reasoning."
]

print("=== Side-by-Side Comparison ===\n")

for i, prompt in enumerate(PROMPTS, 1):
    print(f"Prompt {i}: {prompt}\n")

    # Quantized
    q_out = llm_q.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
        temperature=0.7
    )
    print(f"🔶 Q4_K_M:\n{q_out['choices'][0]['message']['content']}\n")

    # Original
    o_out = pipe(prompt, return_full_text=False)[0]['generated_text']
    print(f"🔷 Original BF16:\n{o_out}\n")
    print("-" * 80)

=== Side-by-Side Comparison ===

Prompt 1: Draft a polite professional email to a client explaining that the project delivery will be delayed by 3 days due to unexpected supply chain issues.

🔶 Q4_K_M:
Subject: Project Delivery Update: Expected Delay Due to Supply Chain Disruptions

Dear [Client's Name],

I hope this email finds you well. I am writing to inform you of an unforeseen delay in the delivery of our project, [Project Name].

Despite our best efforts to adhere to the original timeline, we are experiencing unexpected challenges within our supply chain. These issues have significantly impacted our ability to meet the current delivery date. We understand the importance of this project to your business and regret any inconvenience this may cause.

We are actively working with our suppliers to mitigate the impact and have adjusted our internal schedules to ensure that the project meets our revised delivery date of [New Date]. Our team is committed to ensuring that the quality of t

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 